In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt

In [2]:
#creating dataframes using just closing prices like FRED's format'

sp500 = pd.read_csv('SP500.csv', usecols=['Date','Close'], index_col='Date').round({'Close':2})
sp500.index = pd.to_datetime(sp500.index)

dxy = pd.read_csv('DXY.csv', usecols=['Date','Close'], index_col='Date').round({'Close':2})
dxy.index = pd.to_datetime(dxy.index)

us02y = pd.read_csv('US02Y.csv', usecols=['Date','Close'], index_col='Date')
us02y.index = pd.to_datetime(us02y.index)

vix = pd.read_csv('VIX.csv', usecols=['Date','Close'], index_col='Date').round({'Close':2})
vix.index = pd.to_datetime(vix.index)

In [3]:
market_df = sp500.merge(dxy, left_index=True, right_index=True, how='inner')
market_df = market_df.merge(us02y, left_index=True, right_index=True, how='inner')
market_df = market_df.merge(vix, left_index=True, right_index=True, how='inner', suffixes=('_w','_z'))

market_df = market_df.rename(columns={
    'Close_x':'SP500',
    'Close_y':'DXY',
    'Close_w':'US02Y',
    'Close_z':'VIX'
})

market_df.head()


,SP500,DXY,US02Y,VIX
Date,,,,
1990-01-02,359.69,94.29,7.87,17.24
1990-01-03,358.76,94.42,7.94,18.19
1990-01-04,355.67,92.52,7.92,19.22
1990-01-05,352.20,92.85,7.90,20.11
1990-01-08,353.79,92.05,7.90,20.26


In [4]:
market_df.isna().sum()

SP500     0
DXY       0
US02Y    73
VIX       0
dtype: int64

In [5]:
market_df = market_df.dropna()
print(market_df.isna().sum())
print(market_df.shape)

SP500    0
DXY      0
US02Y    0
VIX      0
dtype: int64
(9111, 4)


In [12]:
#1 week = 5 closes
#1 month = 21 closes
#6 months = 126 closes
# 1 year = 252 closes

log_sp500 = np.log(market_df['SP500'])

market_df['fut_returns_5d'] = (log_sp500.shift(-5) - log_sp500).round(2)
market_df['fut_returns_21d'] = (log_sp500.shift(-21) - log_sp500).round(2)
market_df['fut_returns_126d'] = (log_sp500.shift(-126) - log_sp500).round(2)
market_df['fut_returns_252d'] = (log_sp500.shift(-252) - log_sp500).round(2)

market_df.head()


,SP500,DXY,US02Y,VIX,fut_returns_5d,fut_returns_21d,fut_returns_126d,fut_returns_252d
Date,,,,,,,,
1990-01-02,359.69,94.29,7.87,17.24,-0.03,-0.09,0.00,-0.11
1990-01-03,358.76,94.42,7.94,18.19,-0.03,-0.08,-0.01,-0.13
1990-01-04,355.67,92.52,7.92,19.22,-0.02,-0.07,0.01,-0.12
1990-01-05,352.20,92.85,7.90,20.11,-0.04,-0.07,0.02,-0.12
1990-01-08,353.79,92.05,7.90,20.26,-0.04,-0.06,0.01,-0.12


In [19]:
log_vix = np.log(market_df['VIX'])

market_df['delta_vix_5d'] = (log_vix - log_vix.shift(5)).round(2)
market_df['delta_vix_21d'] = (log_vix - log_vix.shift(21)).round(2)
market_df['delta_vix_126d'] = (log_vix - log_vix.shift(126)).round(2)
market_df['delta_vix_252d'] = (log_vix - log_vix.shift(252)).round(2)

log_dxy = np.log(market_df['DXY'])

market_df['delta_dxy_5d'] = (log_dxy - log_dxy.shift(5)).round(2)
market_df['delta_dxy_21d'] = (log_dxy - log_dxy.shift(21)).round(2)
market_df['delta_dxy_126d'] = (log_dxy - log_dxy.shift(126)).round(2)
market_df['delta_dxy_252d'] = (log_dxy - log_dxy.shift(252)).round(2)

market_df['delta_US02Y_5d'] = (market_df['US02Y'] - market_df['US02Y'].shift(5)).round(2)
market_df['delta_US02Y_21d'] = (market_df['US02Y'] - market_df['US02Y'].shift(21)).round(2)
market_df['delta_US02Y_126d'] = (market_df['US02Y'] - market_df['US02Y'].shift(126)).round(2)
market_df['delta_US02Y_252d'] = (market_df['US02Y'] - market_df['US02Y'].shift(252)).round(2)

market_df.shape

(9111, 20)

In [21]:
market_df = market_df.dropna()
market_df.shape

(8607, 20)